# VLM PDF Extractor — Colab Demo

Upload a PDF, run the full extraction pipeline, and view structured JSON output.

**Requirements:** Google Colab with T4 GPU runtime.

In [1]:
# Cell 1: Install dependencies
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes qwen-vl-utils[decord] datasets pillow
!pip install -q PyMuPDF opencv-python-headless gradio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 28.7 MB/s eta 0:00:00


In [2]:
#If using llama vision model
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
# Cell 2: Clone repo (skip if already cloned)
import os
REPO_DIR = "/content/VLM-PDF-Text_Extractor"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/RahulReadd/VLM-PDF-Text_Extractor.git
os.chdir(REPO_DIR)
!pwd

Cloning into 'VLM-PDF-Text_Extractor'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 71 (delta 31), reused 61 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 53.19 KiB | 5.32 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/VLM-PDF-Text_Extractor


In [6]:
# Cell 3: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [7]:
# Cell 4: Import pipeline modules
import sys, json
sys.path.insert(0, ".")

from app.pdf_to_image import convert_pdf, load_image, print_diagnostics
from app.extract import DocumentExtractor, parse_json_output
from app.evaluate import (
    evaluate_single, parse_cord_ground_truth
)

print("Modules loaded successfully.")

Modules loaded successfully.


In [8]:
# Cell 5: Load the VLM (change model name as needed)
MODEL_NAME = "qwen3-vl-2b"  # Options: qwen3-vl-2b, qwen3-vl-4b, internvl35-8b-4bit, etc.

extractor = DocumentExtractor(model_name=MODEL_NAME)
print(f"Model '{MODEL_NAME}' loaded and ready.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

DocumentExtractor ready: qwen3-vl-2b
Model 'qwen3-vl-2b' loaded and ready.


In [9]:
# Cell 6: Gradio UI
import gradio as gr
from PIL import Image

def process_pdf(pdf_file):
    """Process an uploaded PDF — unified extraction (all 4 aspects)."""
    if pdf_file is None:
        return "No file uploaded.", None

    images, diagnostics = convert_pdf(pdf_file.name)

    diag_text = []
    for d in diagnostics:
        px = d.final_size[0] * d.final_size[1]
        diag_text.append(
            f"Page {d.page_num}: {d.width_in}\"x{d.height_in}\" ({d.page_type}) "
            f"| DPI={d.optimal_dpi} | skew={d.skew_angle} deg "
            f"| {d.final_size[0]}x{d.final_size[1]} ({px/1e6:.1f}M px)"
        )

    all_results = []
    for i, img in enumerate(images):
        result = extractor.extract(img)  # unified prompt, no task
        all_results.append({
            "page": i + 1,
            "json_valid": result["json_valid"],
            "extraction": result["parsed_json"],
        })

    output = {
        "model": MODEL_NAME,
        "mode": "full_extraction",
        "diagnostics": diag_text,
        "pages": all_results,
    }
    return json.dumps(output, indent=2, ensure_ascii=False), images[0] if images else None


def process_image(image):
    """Process an uploaded image — unified extraction."""
    if image is None:
        return "No image uploaded."
    img = Image.fromarray(image).convert("RGB")
    result = extractor.extract(img)  # unified prompt
    return json.dumps(result["parsed_json"], indent=2, ensure_ascii=False)


with gr.Blocks(title="VLM PDF Extractor") as demo:
    gr.Markdown("# VLM PDF Extractor")
    gr.Markdown(f"**Model:** {MODEL_NAME} | **GPU:** T4")
    gr.Markdown("Extracts **key-value pairs**, **signature detection**, **form field status**, and **receipt data** in a single pass.")

    with gr.Tab("PDF Upload"):
        pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
        pdf_btn = gr.Button("Extract All", variant="primary")
        with gr.Row():
            pdf_preview = gr.Image(label="First Page Preview")
            pdf_output = gr.Textbox(label="Extracted JSON", lines=25)
        pdf_btn.click(process_pdf, [pdf_input], [pdf_output, pdf_preview])

    with gr.Tab("Image Upload"):
        img_input = gr.Image(label="Upload Image")
        img_btn = gr.Button("Extract All", variant="primary")
        img_output = gr.Textbox(label="Extracted JSON", lines=25)
        img_btn.click(process_image, [img_input], [img_output])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cad315d96503279c1b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
